# Predictions02 - Using v2 Models (Enhanced Features)
Loads models trained in **Notebook03** with 31 features:
- H2H last 5 matches
- Recent form (last 5)
- Home vs Away win rate split
- Competition importance (1-4)
- FIFA ranking & rank difference

In [1]:
# ============================================================
# CELL 1 - Load Data & Normalize Country Names
# ============================================================
import pandas as pd
import numpy as np
import joblib

results      = pd.read_csv('../data/results.csv')
shootouts    = pd.read_csv('../data/shootouts.csv')
former_names = pd.read_csv('../data/former_names.csv')
rankings     = pd.read_csv('../dataII/rankings.csv')

results['date']   = pd.to_datetime(results['date'])
shootouts['date'] = pd.to_datetime(shootouts['date'])

name_map = dict(zip(former_names['former'], former_names['current']))
def normalize(name): return name_map.get(name, name)

results['home_team']   = results['home_team'].apply(normalize)
results['away_team']   = results['away_team'].apply(normalize)
shootouts['home_team'] = shootouts['home_team'].apply(normalize)
shootouts['away_team'] = shootouts['away_team'].apply(normalize)
shootouts['winner']    = shootouts['winner'].apply(normalize)

results_sorted = results.dropna(subset=['home_score','away_score']).sort_values('date').reset_index(drop=True)

rank_lookup   = dict(zip(rankings['team'], rankings['fifa_rank']))
points_lookup = dict(zip(rankings['team'], rankings['fifa_points']))

print('Data loaded!')
print('results:', results_sorted.shape, '  rankings:', rankings.shape)

Data loaded!
results: (49448, 9)   rankings: (216, 4)


In [2]:
# ============================================================
# CELL 2 - Helper Functions (same as Notebook03)
# ============================================================

def get_rank(team):
    return rank_lookup.get(team, 200)

def team_stats(team, before_date, n=20):
    past = results_sorted[
        (results_sorted['date'] < before_date) &
        ((results_sorted['home_team'] == team) | (results_sorted['away_team'] == team))
    ].tail(n)
    if len(past) == 0:
        return {'win_rate': 0.5, 'avg_scored': 1.0, 'avg_conceded': 1.0, 'n_matches': 0}
    wins, scored, conceded = [], [], []
    for _, m in past.iterrows():
        if m['home_team'] == team:
            wins.append(1 if m['home_score'] > m['away_score'] else 0)
            scored.append(m['home_score']); conceded.append(m['away_score'])
        else:
            wins.append(1 if m['away_score'] > m['home_score'] else 0)
            scored.append(m['away_score']); conceded.append(m['home_score'])
    return {'win_rate': np.mean(wins), 'avg_scored': np.mean(scored),
            'avg_conceded': np.mean(conceded), 'n_matches': len(past)}

def team_stats_split(team, before_date, n=20):
    past_home = results_sorted[
        (results_sorted['date'] < before_date) & (results_sorted['home_team'] == team)
    ].tail(n)
    past_away = results_sorted[
        (results_sorted['date'] < before_date) & (results_sorted['away_team'] == team)
    ].tail(n)
    home_wr = (past_home['home_score'] > past_home['away_score']).mean() if len(past_home) > 0 else 0.5
    away_wr = (past_away['away_score'] > past_away['home_score']).mean() if len(past_away) > 0 else 0.5
    return float(home_wr), float(away_wr)

def recent_form(team, before_date, n=5):
    s = team_stats(team, before_date, n=n)
    return s['win_rate'], s['avg_scored'], s['avg_conceded']

def h2h_stats(home_team, away_team, before_date, n=5):
    past = results_sorted[
        (results_sorted['date'] < before_date) &
        (
            ((results_sorted['home_team'] == home_team) & (results_sorted['away_team'] == away_team)) |
            ((results_sorted['home_team'] == away_team) & (results_sorted['away_team'] == home_team))
        )
    ].tail(n)
    if len(past) == 0:
        return {'h2h_home_win_rate': 0.5, 'h2h_avg_home_scored': 1.0, 'h2h_avg_away_scored': 1.0, 'h2h_n': 0}
    home_wins, home_scored, away_scored = [], [], []
    for _, m in past.iterrows():
        if m['home_team'] == home_team:
            home_wins.append(1 if m['home_score'] > m['away_score'] else 0)
            home_scored.append(m['home_score']); away_scored.append(m['away_score'])
        else:
            home_wins.append(1 if m['away_score'] > m['home_score'] else 0)
            home_scored.append(m['away_score']); away_scored.append(m['home_score'])
    return {
        'h2h_home_win_rate':   np.mean(home_wins),
        'h2h_avg_home_scored': np.mean(home_scored),
        'h2h_avg_away_scored': np.mean(away_scored),
        'h2h_n':               len(past)
    }

def comp_importance(tournament):
    t = str(tournament).lower()
    if 'fifa world cup' in t:                                              return 4
    if any(x in t for x in ['euro', 'copa america', 'asian cup',
                              'africa cup', 'gold cup', 'nations cup']):   return 3
    if any(x in t for x in ['qualifier', 'qualification']):               return 2
    return 1

def shootout_win_rate(team, before_date):
    past = shootouts[
        (shootouts['date'] < before_date) &
        ((shootouts['home_team'] == team) | (shootouts['away_team'] == team))
    ]
    if len(past) == 0: return 0.5
    return (past['winner'] == team).sum() / len(past)

print('Helper functions ready!')

Helper functions ready!


In [3]:
# ============================================================
# CELL 3 - Load v2 Models
# ============================================================
model_outcome    = joblib.load('../dataII/model_outcome_v2.pkl')
model_home_score = joblib.load('../dataII/model_home_score_v2.pkl')
model_away_score = joblib.load('../dataII/model_away_score_v2.pkl')

FEATURES = [
    'is_neutral', 'year', 'month', 'comp_importance',
    'home_win_rate', 'home_avg_scored', 'home_avg_conceded', 'home_n_matches', 'home_shootout_wr',
    'home_home_win_rate', 'home_away_win_rate',
    'home_form_win_rate', 'home_form_avg_scored', 'home_form_avg_conceded',
    'away_win_rate', 'away_avg_scored', 'away_avg_conceded', 'away_n_matches', 'away_shootout_wr',
    'away_home_win_rate', 'away_away_win_rate',
    'away_form_win_rate', 'away_form_avg_scored', 'away_form_avg_conceded',
    'h2h_home_win_rate', 'h2h_avg_home_scored', 'h2h_avg_away_scored', 'h2h_n',
    'home_fifa_rank', 'away_fifa_rank', 'rank_diff',
]

print(f'Loaded: {type(model_outcome).__name__} (outcome)')
print(f'Loaded: {type(model_home_score).__name__} (home score)')
print(f'Loaded: {type(model_away_score).__name__} (away score)')
print(f'Feature count: {len(FEATURES)}')

Loaded: GradientBoostingClassifier (outcome)
Loaded: GradientBoostingRegressor (home score)
Loaded: GradientBoostingRegressor (away score)
Feature count: 31


In [13]:
# ============================================================
# CELL 4 - Predict Function (v2)
# ============================================================
def predict(home_team, away_team, is_neutral=0, tournament='Friendly'):
    today = pd.Timestamp.today()

    h           = team_stats(home_team, today)
    a           = team_stats(away_team, today)
    h_hw, h_aw  = team_stats_split(home_team, today)
    a_hw, a_aw  = team_stats_split(away_team, today)
    h_fwr, h_fsc, h_fcc = recent_form(home_team, today)
    a_fwr, a_fsc, a_fcc = recent_form(away_team, today)
    h2h         = h2h_stats(home_team, away_team, today)
    home_rank   = get_rank(home_team)
    away_rank   = get_rank(away_team)

    row = pd.DataFrame([{
        'is_neutral':              is_neutral,
        'year':                    today.year,
        'month':                   today.month,
        'comp_importance':         comp_importance(tournament),
        'home_win_rate':           h['win_rate'],
        'home_avg_scored':         h['avg_scored'],
        'home_avg_conceded':       h['avg_conceded'],
        'home_n_matches':          h['n_matches'],
        'home_shootout_wr':        shootout_win_rate(home_team, today),
        'home_home_win_rate':      h_hw,
        'home_away_win_rate':      h_aw,
        'home_form_win_rate':      h_fwr,
        'home_form_avg_scored':    h_fsc,
        'home_form_avg_conceded':  h_fcc,
        'away_win_rate':           a['win_rate'],
        'away_avg_scored':         a['avg_scored'],
        'away_avg_conceded':       a['avg_conceded'],
        'away_n_matches':          a['n_matches'],
        'away_shootout_wr':        shootout_win_rate(away_team, today),
        'away_home_win_rate':      a_hw,
        'away_away_win_rate':      a_aw,
        'away_form_win_rate':      a_fwr,
        'away_form_avg_scored':    a_fsc,
        'away_form_avg_conceded':  a_fcc,
        'h2h_home_win_rate':       h2h['h2h_home_win_rate'],
        'h2h_avg_home_scored':     h2h['h2h_avg_home_scored'],
        'h2h_avg_away_scored':     h2h['h2h_avg_away_scored'],
        'h2h_n':                   h2h['h2h_n'],
        'home_fifa_rank':          home_rank,
        'away_fifa_rank':          away_rank,
        'rank_diff':               away_rank - home_rank,
    }])

    outcome = model_outcome.predict(row)[0]
    proba   = model_outcome.predict_proba(row)[0]
    home_g  = max(0, int(round(model_home_score.predict(row)[0])))
    away_g  = max(0, int(round(model_away_score.predict(row)[0])))
    label   = {0: 'Home Win', 1: 'Draw', 2: 'Away Win'}

    venue = 'Neutral' if is_neutral else f'{home_team} home ground'
    print(f'  {home_team}  vs  {away_team}')
    # print(f'  Tournament     : {tournament}  (importance={comp_importance(tournament)})')
    # print(f'  Venue          : {venue}')
    # print(f'  FIFA Ranks     : {home_team} #{int(home_rank)}  vs  {away_team} #{int(away_rank)}')
    print(f'  Predicted Score: {home_team} {home_g} - {away_g} {away_team}')
    print(f'  Outcome        : {label[outcome]}')
    print(f'  {home_team} win  : {proba[0]*100:.1f}%')
    print(f'  Draw           : {proba[1]*100:.1f}%')
    print(f'  {away_team} win  : {proba[2]*100:.1f}%')
    print()

print('predict() ready!')

predict() ready!


In [14]:
# ============================================================
# CELL 5 - Run Predictions
# ============================================================
# predict('Scotland',    'Morocco',  is_neutral=1, tournament='FIFA World Cup')
# predict('France',    'England',    is_neutral=1, tournament='FIFA World Cup')
# predict('Spain',     'Cape Verde',    is_neutral=1, tournament='FIFA World Cup')
# predict('Norway',    'Senegal',    is_neutral=1, tournament='FIFA World Cup')
# predict('Argentina', 'Iran',       is_neutral=1, tournament='FIFA World Cup')
# predict('Jordan',    'Algeria',    is_neutral=1, tournament='FIFA World Cup')
predict('Portugal',  'Uzbekistan', is_neutral=1, tournament='FIFA World Cup')
predict('England',   'Ghana',      is_neutral=1, tournament='FIFA World Cup')

  Portugal  vs  Uzbekistan
  Predicted Score: Portugal 2 - 1 Uzbekistan
  Outcome        : Home Win
  Portugal win  : 71.4%
  Draw           : 20.1%
  Uzbekistan win  : 8.5%

  England  vs  Ghana
  Predicted Score: England 2 - 1 Ghana
  Outcome        : Home Win
  England win  : 74.0%
  Draw           : 19.3%
  Ghana win  : 6.6%



In [15]:
predict('Panama', 'Croatia', is_neutral=1, tournament='FIFA World Cup')
predict('Colombia', 'DR Congo', is_neutral=1, tournament='FIFA World Cup')

  Panama  vs  Croatia
  Predicted Score: Panama 1 - 2 Croatia
  Outcome        : Away Win
  Panama win  : 21.4%
  Draw           : 26.3%
  Croatia win  : 52.3%

  Colombia  vs  DR Congo
  Predicted Score: Colombia 2 - 1 DR Congo
  Outcome        : Home Win
  Colombia win  : 57.5%
  Draw           : 22.1%
  DR Congo win  : 20.4%

